# Sequential Quality-Aware Refinement LDM for CT→MRI

## 핵심 아이디어

### Two-Stage Architecture:
```
Stage 1 (Frozen LDM):
  Sequential CT slices → Draft MRI slices
  
Stage 2 (Quality-Aware Refiner):
  1. 양옆 ±5 슬라이스의 draft 중 좋은 것 5개 선택 (Learnable)
  2. Selected drafts → Context features
  3. Center draft + CT + Context → Refined MRI (Attention U-Net)
```

## 장점
- ✅ Stage 1은 이미 작동 (기존 2D LDM)
- ✅ Learnable quality selector로 최적 draft 자동 선택
- ✅ Attention mechanism으로 중요 정보 집중
- ✅ Sequential 특성 완벽 활용
- ✅ Progressive training으로 안정적 학습

## 1. Setup & Imports

In [2]:
import sys
import os
from pathlib import Path

# LDM 경로 추가
ldm_path = os.path.expanduser('~/CT2MRI/diffusion_model/ldm')
if ldm_path not in sys.path:
    sys.path.insert(0, ldm_path)

print(f"Added to path: {ldm_path}")

Added to path: /home/youkyoung/CT2MRI/diffusion_model/ldm


In [3]:
import glob
import time
import warnings
import copy
from typing import List, Tuple, Dict, Optional
from itertools import chain

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# LDM imports
from ldm.modules.diffusionmodules.openaimodel import UNetModel
from ldm.modules.diffusionmodules.util import (
    make_beta_schedule, 
    make_ddim_timesteps, 
    make_ddim_sampling_parameters
)
from ldm.models.autoencoder import AutoencoderKL

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1+cu124
CUDA available: True
CUDA device: NVIDIA L40S


## 2. Configuration

In [4]:
CFG = {
    # ========== Device ==========
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
    
    # ========== Data ==========
    "data_root": "../data/processed_with_skullstrip",
    "img_size": 256,
    "train_ratio": 0.9,
    
    # ========== Sequential Context ==========
    "context_range": 5,  # ±5 slices (total 11)
    "num_selected": 5,   # Select top 5 from 11
    
    # ========== Pretrained Model ==========
    "pretrained_ckpt": "./ckpt_ct2mri_b0_ldm_strip/last.pt",
    
    # ========== Model Architecture ==========
    "z_channels": 4,
    
    # ========== Training ==========
    "batch_size": 16,
    "num_workers": 4,
    "epochs": 100,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    
    # ========== Progressive Training ==========
    "phase1_epochs": 20,  # Reconstruction only
    "phase2_epochs": 50,  # + Perceptual
    "phase3_epochs": 100, # Full loss
    
    # ========== Diffusion Sampling ==========
    "num_timesteps": 1000,
    "beta_schedule": "cosine",
    "ddim_steps": 50,
    "ddim_eta": 0.0,
    
    # ========== Checkpointing ==========
    "save_dir": "./ckpt_sequential_refinement_ldm",
    "save_every": 5,
}

# Set seed
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

# Create save directory
Path(CFG["save_dir"]).mkdir(parents=True, exist_ok=True)

print("Configuration:")
for k, v in CFG.items():
    print(f"  {k:25s}: {v}")

Configuration:
  device                   : cuda
  seed                     : 42
  data_root                : ../data/processed_with_skullstrip
  img_size                 : 256
  train_ratio              : 0.9
  context_range            : 5
  num_selected             : 5
  pretrained_ckpt          : ./ckpt_ct2mri_b0_ldm_strip/last.pt
  z_channels               : 4
  batch_size               : 16
  num_workers              : 4
  epochs                   : 100
  lr                       : 0.0001
  weight_decay             : 0.0001
  grad_clip                : 1.0
  phase1_epochs            : 20
  phase2_epochs            : 50
  phase3_epochs            : 100
  num_timesteps            : 1000
  beta_schedule            : cosine
  ddim_steps               : 50
  ddim_eta                 : 0.0
  save_dir                 : ./ckpt_sequential_refinement_ldm
  save_every               : 5


## 3. Dataset for Sequential Processing

In [5]:
class SequentialRefinementDataset(Dataset):
    """
    Dataset for sequential refinement
    Returns: center slice + ±K neighbor slices (total 2K+1)
    """
    
    def __init__(
        self, 
        data_root: str, 
        context_range: int = 5,
        img_size: int = 256,
        mode: str = 'train',
        train_ratio: float = 0.9
    ):
        self.data_root = Path(data_root)
        self.context_range = context_range
        self.img_size = img_size
        self.mode = mode
        
        print(f"\n[{mode.upper()}] Collecting patient slices from {data_root}...")
        self.patient_slices = self._collect_patient_slices()
        print(f"  Found {len(self.patient_slices)} patients")
        
        # Train/Val split
        patients = sorted(self.patient_slices.keys())
        n_train = int(len(patients) * train_ratio)
        
        if mode == 'train':
            self.patients = patients[:n_train]
        else:
            self.patients = patients[n_train:]
        
        # Generate samples (only center slices that have enough context)
        self.samples = []
        for patient in self.patients:
            slices = self.patient_slices[patient]
            n_slices = len(slices['ct'])
            
            # Only use slices with full context
            for i in range(context_range, n_slices - context_range):
                self.samples.append((patient, i))
        
        print(f"  {len(self.samples)} samples from {len(self.patients)} patients")
        print(f"  Context range: ±{context_range} slices\n")
    
    def _collect_patient_slices(self) -> Dict:
        """환자별로 CT, MRI 슬라이스 수집"""
        patient_data = {}
        
        for group in ['bleed', 'non_bleed']:
            group_dir = self.data_root / group
            if not group_dir.exists():
                continue
                
            for patient_dir in sorted(group_dir.iterdir()):
                if not patient_dir.is_dir():
                    continue
                
                patient_name = f"{group}_{patient_dir.name}"
                
                ct_dir = patient_dir / 'ct_axial'
                mri_dir = patient_dir / 'mri_b0_axial'
                
                if not (ct_dir.exists() and mri_dir.exists()):
                    continue
                
                ct_files = sorted(ct_dir.glob('*.png'))
                mri_files = sorted(mri_dir.glob('*.png'))
                
                ct_dict = {f.stem: f for f in ct_files}
                mri_dict = {f.stem: f for f in mri_files}
                
                common_stems = sorted(set(ct_dict.keys()) & set(mri_dict.keys()))
                
                # Need at least 2*context_range + 1 slices
                if len(common_stems) < 2 * self.context_range + 3:
                    continue
                
                patient_data[patient_name] = {
                    'ct': [ct_dict[stem] for stem in common_stems],
                    'mri': [mri_dict[stem] for stem in common_stems]
                }
        
        return patient_data
    
    def _load_image(self, path: Path) -> np.ndarray:
        """이미지 로드 및 전처리"""
        img = Image.open(path).convert('L').resize(
            (self.img_size, self.img_size), Image.BICUBIC
        )
        return np.array(img, dtype=np.float32) / 255.0
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        patient, center_idx = self.samples[idx]
        slices = self.patient_slices[patient]
        
        # Load center slice
        ct_center = self._load_image(slices['ct'][center_idx])
        mri_center = self._load_image(slices['mri'][center_idx])
        
        # Load context slices (±K)
        ct_sequence = []
        mri_sequence = []
        
        for offset in range(-self.context_range, self.context_range + 1):
            idx_neighbor = center_idx + offset
            
            ct_neighbor = self._load_image(slices['ct'][idx_neighbor])
            mri_neighbor = self._load_image(slices['mri'][idx_neighbor])
            
            ct_sequence.append(ct_neighbor)
            mri_sequence.append(mri_neighbor)
        
        # Convert to tensors: [0,1] → [-1,1]
        ct_sequence_t = torch.stack([
            torch.from_numpy(ct).unsqueeze(0) * 2 - 1
            for ct in ct_sequence
        ], dim=0)  # (2K+1, 1, H, W)
        
        mri_sequence_t = torch.stack([
            torch.from_numpy(mri).unsqueeze(0) * 2 - 1
            for mri in mri_sequence
        ], dim=0)  # (2K+1, 1, H, W)
        
        mri_center_t = torch.from_numpy(mri_center).unsqueeze(0) * 2 - 1
        
        return {
            'ct_sequence': ct_sequence_t,      # (11, 1, 256, 256)
            'mri_sequence': mri_sequence_t,    # (11, 1, 256, 256) - for quality evaluation
            'mri_center': mri_center_t,        # (1, 256, 256) - target
            'patient': patient,
            'slice_idx': center_idx
        }

## 4. Model Components

In [6]:
class SelfAttention(nn.Module):
    """Self-attention module for feature refinement"""
    
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        
        self.q = nn.Conv2d(channels, channels // 8, 1)
        self.k = nn.Conv2d(channels, channels // 8, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        
        self.gamma = nn.Parameter(torch.zeros(1))
    
    def forward(self, x):
        B, C, H, W = x.shape
        
        q = self.q(x).view(B, -1, H * W).permute(0, 2, 1)  # (B, HW, C')
        k = self.k(x).view(B, -1, H * W)  # (B, C', HW)
        v = self.v(x).view(B, -1, H * W)  # (B, C, HW)
        
        attn = torch.softmax(torch.bmm(q, k), dim=-1)  # (B, HW, HW)
        out = torch.bmm(v, attn.permute(0, 2, 1))  # (B, C, HW)
        out = out.view(B, C, H, W)
        
        return x + self.gamma * out


class CrossAttention(nn.Module):
    """Cross-attention between query and key/value"""
    
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        
        self.q = nn.Conv2d(channels, channels // 8, 1)
        self.k = nn.Conv2d(channels, channels // 8, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        
        self.gamma = nn.Parameter(torch.zeros(1))
    
    def forward(self, query, key_value):
        B, C, H, W = query.shape
        
        q = self.q(query).view(B, -1, H * W).permute(0, 2, 1)  # (B, HW, C')
        k = self.k(key_value).view(B, -1, H * W)  # (B, C', HW)
        v = self.v(key_value).view(B, -1, H * W)  # (B, C, HW)
        
        attn = torch.softmax(torch.bmm(q, k), dim=-1)  # (B, HW, HW)
        out = torch.bmm(v, attn.permute(0, 2, 1))  # (B, C, HW)
        out = out.view(B, C, H, W)
        
        return query + self.gamma * out


print("✓ Attention modules defined")

✓ Attention modules defined


In [7]:
class LearnableQualitySelector(nn.Module):
    """
    Neural network that learns to select high-quality drafts
    
    Input: Draft MRI + corresponding CT
    Output: Quality score (higher = better)
    """
    
    def __init__(self):
        super().__init__()
        
        # Quality assessment network
        self.features = nn.Sequential(
            # Input: draft(1) + ct(1) = 2 channels
            nn.Conv2d(2, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # /2
            
            nn.Conv2d(32, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # /4
            
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # /8
            
            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
            nn.Sigmoid()  # Quality score in [0, 1]
        )
    
    def forward(self, draft_sequence, ct_sequence, ground_truth=None):
        """
        Args:
            draft_sequence: (B, N, 1, H, W) - N draft MRIs
            ct_sequence: (B, N, 1, H, W) - Corresponding CTs
            ground_truth: (B, 1, H, W) - Ground truth MRI (training only)
        
        Returns:
            quality_scores: (B, N) - Quality scores for each draft
            quality_loss: Scalar (training only)
        """
        B, N = draft_sequence.shape[:2]
        
        # Compute quality scores for all drafts
        quality_scores = []
        
        for n in range(N):
            draft = draft_sequence[:, n]  # (B, 1, H, W)
            ct = ct_sequence[:, n]  # (B, 1, H, W)
            
            # Concatenate draft and CT
            combined = torch.cat([draft, ct], dim=1)  # (B, 2, H, W)
            
            # Extract features and predict quality
            feat = self.features(combined)
            score = self.classifier(feat)  # (B, 1)
            
            quality_scores.append(score.squeeze(1))  # (B,)
        
        quality_scores = torch.stack(quality_scores, dim=1)  # (B, N)
        
        # Training: compute quality loss
        quality_loss = torch.tensor(0.0, device=quality_scores.device)
        
        if self.training and ground_truth is not None:
            # Compute true quality: inverse of MSE to ground truth
            true_quality = []
            
            for n in range(N):
                draft = draft_sequence[:, n]
                mse = F.mse_loss(
                    draft, 
                    ground_truth,
                    reduction='none'
                ).mean(dim=(1, 2, 3))  # (B,)
                
                # Convert MSE to quality score (lower MSE = higher quality)
                # Use sigmoid to bound in [0, 1]
                quality = torch.sigmoid(-10 * mse)  # Scale factor 10
                true_quality.append(quality)
            
            true_quality = torch.stack(true_quality, dim=1)  # (B, N)
            
            # MSE loss between predicted and true quality
            quality_loss = F.mse_loss(quality_scores, true_quality)
        
        return quality_scores, quality_loss


print("✓ LearnableQualitySelector defined")

✓ LearnableQualitySelector defined


In [8]:
class AttentionUNet(nn.Module):
    """
    U-Net with Self and Cross-Attention mechanisms
    """
    
    def __init__(self, in_channels, out_channels=1):
        super().__init__()
        
        # Encoder
        self.enc1 = self._block(in_channels, 64)
        self.enc2 = self._block(64, 128)
        self.enc3 = self._block(128, 256)
        self.enc4 = self._block(256, 512)
        
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck with self-attention
        self.bottleneck = nn.Sequential(
            self._block(512, 1024),
            SelfAttention(1024),
        )
        
        # Decoder with cross-attention
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4 = self._block(1024, 512)
        self.attn4 = CrossAttention(512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = self._block(512, 256)
        self.attn3 = CrossAttention(256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = self._block(256, 128)
        
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = self._block(128, 64)
        
        # Output layer
        self.out = nn.Conv2d(64, out_channels, 1)
    
    def _block(self, in_ch, out_ch):
        """Conv block: Conv-BN-ReLU x2"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)  # (B, 64, 256, 256)
        e2 = self.enc2(self.pool(e1))  # (B, 128, 128, 128)
        e3 = self.enc3(self.pool(e2))  # (B, 256, 64, 64)
        e4 = self.enc4(self.pool(e3))  # (B, 512, 32, 32)
        
        # Bottleneck
        b = self.bottleneck(self.pool(e4))  # (B, 1024, 16, 16)
        
        # Decoder with skip connections and cross-attention
        d4 = self.up4(b)  # (B, 512, 32, 32)
        d4 = torch.cat([d4, e4], dim=1)  # (B, 1024, 32, 32)
        d4 = self.dec4(d4)  # (B, 512, 32, 32)
        d4 = self.attn4(d4, e4)  # Cross-attention
        
        d3 = self.up3(d4)  # (B, 256, 64, 64)
        d3 = torch.cat([d3, e3], dim=1)  # (B, 512, 64, 64)
        d3 = self.dec3(d3)  # (B, 256, 64, 64)
        d3 = self.attn3(d3, e3)  # Cross-attention
        
        d2 = self.up2(d3)  # (B, 128, 128, 128)
        d2 = torch.cat([d2, e2], dim=1)  # (B, 256, 128, 128)
        d2 = self.dec2(d2)  # (B, 128, 128, 128)
        
        d1 = self.up1(d2)  # (B, 64, 256, 256)
        d1 = torch.cat([d1, e1], dim=1)  # (B, 128, 256, 256)
        d1 = self.dec1(d1)  # (B, 64, 256, 256)
        
        out = self.out(d1)  # (B, 1, 256, 256)
        
        return out


print("✓ AttentionUNet defined")

✓ AttentionUNet defined


In [9]:
class SequentialRefinementLDM(nn.Module):
    """
    Two-Stage Sequential Refinement LDM
    
    Stage 1 (Frozen): Pretrained 2D LDM generates draft MRIs
    Stage 2 (Trainable): Quality-aware refinement with attention
    """
    
    def __init__(
        self,
        vae: nn.Module,
        unet: nn.Module,
        schedule: dict,
        num_selected: int = 5,
        ddim_steps: int = 50,
        ddim_eta: float = 0.0
    ):
        super().__init__()
        
        self.num_selected = num_selected
        self.ddim_steps = ddim_steps
        self.ddim_eta = ddim_eta
        self.schedule = schedule
        
        print("\nBuilding Sequential Refinement LDM...")
        
        # ===== Stage 1: Frozen LDM =====
        self.vae = vae
        self.unet = unet
        
        self.vae.eval()
        self.unet.eval()
        
        for p in chain(self.vae.parameters(), self.unet.parameters()):
            p.requires_grad = False
        
        print("  ✓ Stage 1: Frozen LDM (VAE + UNet)")
        
        # ===== Check and fix UNet output channels =====
        self.z_channels = 4  # VAE latent channels
        
        # Get UNet output channels
        unet_out_channels = None
        for module in reversed(list(self.unet.out.modules())):
            if isinstance(module, nn.Conv2d):
                unet_out_channels = module.out_channels
                break
        
        print(f"  UNet output channels: {unet_out_channels}")
        
        # Add projection if needed
        if unet_out_channels != self.z_channels:
            print(f"  Adding output projection: {unet_out_channels} → {self.z_channels}")
            self.unet_output_proj = nn.Conv2d(unet_out_channels, self.z_channels, 1).to(self.unet.out[0].weight.device)
            # Initialize with simple mapping
            with torch.no_grad():
                nn.init.kaiming_normal_(self.unet_output_proj.weight)
                nn.init.zeros_(self.unet_output_proj.bias)
            self.unet_output_proj.eval()
            for p in self.unet_output_proj.parameters():
                p.requires_grad = False
        else:
            self.unet_output_proj = None
        
        # ===== Stage 2 Components =====
        
        # Quality Selector
        self.quality_selector = LearnableQualitySelector()
        print("  ✓ Quality Selector (Learnable)")
        
        # Context Encoder: Selected drafts → context features
        self.context_encoder = nn.Sequential(
            nn.Conv2d(num_selected, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        print("  ✓ Context Encoder")
        
        # Attention-based Refiner
        # Input: draft(1) + CT(1) + context(256) = 258 channels
        self.refiner = AttentionUNet(in_channels=258, out_channels=1)
        print("  ✓ Attention-based Refiner")
        
        # Statistics
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"\nModel parameters:")
        print(f"  Trainable: {trainable:,} ({trainable/total*100:.1f}%)")
        print(f"  Total: {total:,}")
    
    @torch.no_grad()
    def generate_draft_sequence(self, ct_sequence):
        """
        Stage 1: Generate draft MRIs using frozen LDM
        
        Args:
            ct_sequence: (B, N, 1, H, W)
        
        Returns:
            draft_sequence: (B, N, 1, H, W)
        """
        B, N = ct_sequence.shape[:2]
        device = ct_sequence.device
        
        draft_sequence = []
        
        for n in range(N):
            ct_slice = ct_sequence[:, n]  # (B, 1, H, W)
            
            # Encode CT to latent
            z_ct = self.vae.encode(ct_slice).sample()  # (B, 4, 32, 32)
            
            # DDIM sampling
            draft = self._ddim_sample(z_ct, device)
            
            draft_sequence.append(draft)
        
        return torch.stack(draft_sequence, dim=1)  # (B, N, 1, H, W)
    
    @torch.no_grad()
    def _ddim_sample(self, z_condition, device):
        """DDIM sampling for a single slice"""
        B = z_condition.shape[0]
        
        # Start from random noise
        z = torch.randn_like(z_condition)
        
        # DDIM timesteps
        ts = torch.tensor(
            make_ddim_timesteps('uniform', self.ddim_steps, len(self.schedule['betas']))
        ) - 1
        
        sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(
            self.schedule['a_bar'].cpu().numpy(),
            ts.numpy(),
            self.ddim_eta
        )
        
        sigmas = torch.from_numpy(sigmas).float().to(device)
        alphas = torch.from_numpy(alphas).float().to(device)
        alphas_prev = torch.from_numpy(alphas_prev).float().to(device)
        
        # Sampling loop
        for i in reversed(range(len(ts))):
            t = int(ts[i])
            t_batch = torch.full((B,), t, device=device, dtype=torch.long)
            
            # Predict noise
            eps = self.unet(z, t_batch, None)
            
            # Apply output projection if needed
            if self.unet_output_proj is not None:
                eps = self.unet_output_proj(eps)
            
            # DDIM update
            a_t = alphas[i]
            a_prev = alphas_prev[i]
            s_t = sigmas[i]
            
            z0_pred = (z - torch.sqrt(1.0 - a_t) * eps) / torch.sqrt(a_t)
            dir_term = torch.sqrt(1.0 - a_prev - s_t ** 2) * eps
            noise = s_t * torch.randn_like(z) if i > 0 else 0.0
            
            z = torch.sqrt(a_prev) * z0_pred + dir_term + noise
        
        # Decode to image
        draft = self.vae.decode(z)
        
        return draft
    
    def forward(self, ct_sequence, mri_center=None, return_intermediates=False):
        """
        Full forward pass
        
        Args:
            ct_sequence: (B, N, 1, H, W) - Sequential CT slices
            mri_center: (B, 1, H, W) - Ground truth (training only)
            return_intermediates: Return draft and selected indices
        
        Returns:
            refined_mri: (B, 1, H, W)
            (optional) draft_center, quality_loss, selected_indices
        """
        B, N = ct_sequence.shape[:2]
        
        # ===== Stage 1: Generate drafts =====
        draft_sequence = self.generate_draft_sequence(ct_sequence)
        # (B, N, 1, H, W)
        
        center_idx = N // 2
        draft_center = draft_sequence[:, center_idx]  # (B, 1, H, W)
        ct_center = ct_sequence[:, center_idx]  # (B, 1, H, W)
        
        # ===== Stage 2-1: Select good drafts =====
        quality_scores, quality_loss = self.quality_selector(
            draft_sequence,
            ct_sequence,
            ground_truth=mri_center
        )
        # quality_scores: (B, N)
        
        # Select top-K drafts
        _, top_indices = torch.topk(quality_scores, k=self.num_selected, dim=1)
        # top_indices: (B, K)
        
        # Gather selected drafts
        selected_drafts = []
        for b in range(B):
            selected = draft_sequence[b, top_indices[b]]  # (K, 1, H, W)
            selected_drafts.append(selected)
        selected_drafts = torch.stack(selected_drafts, dim=0)  # (B, K, 1, H, W)
        
        # ===== Stage 2-2: Encode context =====
        selected_drafts_flat = selected_drafts.squeeze(2)  # (B, K, H, W)
        context_features = self.context_encoder(selected_drafts_flat)  # (B, 256, H, W)
        
        # ===== Stage 2-3: Refine =====
        refiner_input = torch.cat([
            draft_center,      # (B, 1, H, W)
            ct_center,         # (B, 1, H, W)
            context_features   # (B, 256, H, W)
        ], dim=1)  # (B, 258, H, W)
        
        refined_mri = self.refiner(refiner_input)  # (B, 1, H, W)
        
        if return_intermediates:
            return refined_mri, draft_center, quality_loss, top_indices
        
        return refined_mri, draft_center, quality_loss


print("✓ SequentialRefinementLDM defined")

✓ SequentialRefinementLDM defined


## 5. Loss Function

In [10]:
class RefinementLoss(nn.Module):
    """
    Multi-component loss for refinement training
    """
    
    def __init__(self):
        super().__init__()
        
        # Perceptual loss network (VGG16)
        vgg = models.vgg16(pretrained=True)
        self.vgg_features = vgg.features[:16].eval()
        
        for p in self.vgg_features.parameters():
            p.requires_grad = False
        
        print("✓ RefinementLoss with VGG16 perceptual loss")
    
    def forward(self, refined, draft, target, quality_loss, epoch=0):
        """
        Args:
            refined: Refined MRI (B, 1, H, W)
            draft: Draft MRI (B, 1, H, W)
            target: Ground truth (B, 1, H, W)
            quality_loss: From quality selector
            epoch: Current epoch (for progressive training)
        """
        
        # 1. Reconstruction loss (L1 + L2)
        l1_loss = F.l1_loss(refined, target)
        l2_loss = F.mse_loss(refined, target)
        recon_loss = 0.7 * l1_loss + 0.3 * l2_loss
        
        # 2. Perceptual loss (VGG features)
        perc_loss = torch.tensor(0.0, device=refined.device)
        
        if epoch >= 20:  # Start from phase 2
            # Convert grayscale to 3-channel
            refined_3ch = refined.repeat(1, 3, 1, 1)
            target_3ch = target.repeat(1, 3, 1, 1)
            
            # Normalize to ImageNet stats
            mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(refined.device)
            std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(refined.device)
            
            refined_norm = (refined_3ch - mean) / std
            target_norm = (target_3ch - mean) / std
            
            # Extract features
            with torch.no_grad():
                target_feat = self.vgg_features(target_norm)
            
            refined_feat = self.vgg_features(refined_norm)
            
            perc_loss = F.mse_loss(refined_feat, target_feat)
        
        # 3. Improvement loss (refined should be better than draft)
        improve_loss = torch.tensor(0.0, device=refined.device)
        
        if epoch >= 20:  # Start from phase 2
            draft_error = F.l1_loss(draft, target, reduction='none').mean(dim=(1,2,3))
            refined_error = F.l1_loss(refined, target, reduction='none').mean(dim=(1,2,3))
            
            # Penalize if refined is worse than draft
            improve_loss = F.relu(refined_error - draft_error + 0.01).mean()
        
        # 4. Progressive weighting
        if epoch < 20:
            # Phase 1: Reconstruction only
            weights = {'recon': 1.0, 'perc': 0.0, 'improve': 0.0, 'quality': 0.1}
        elif epoch < 50:
            # Phase 2: Add perceptual
            weights = {'recon': 1.0, 'perc': 0.3, 'improve': 0.1, 'quality': 0.2}
        else:
            # Phase 3: Full loss
            weights = {'recon': 1.0, 'perc': 0.5, 'improve': 0.2, 'quality': 0.3}
        
        # Total loss
        total_loss = (
            weights['recon'] * recon_loss +
            weights['perc'] * perc_loss +
            weights['improve'] * improve_loss +
            weights['quality'] * quality_loss
        )
        
        return {
            'total': total_loss,
            'recon': recon_loss,
            'l1': l1_loss,
            'l2': l2_loss,
            'perceptual': perc_loss,
            'improvement': improve_loss,
            'quality': quality_loss
        }


print("✓ RefinementLoss defined")

✓ RefinementLoss defined


## 6. Build Model from Pretrained

In [11]:
def build_sequential_refinement_model(cfg):
    """Build model from pretrained 2D LDM checkpoint"""
    device = cfg["device"]
    ckpt_path = cfg["pretrained_ckpt"]
    
    print(f"\nLoading pretrained 2D LDM from: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location='cpu')
    
    # 1. VAE
    print("Building VAE...")
    vae = AutoencoderKL(
        ddconfig={
            'double_z': True,
            'z_channels': cfg['z_channels'],
            'resolution': cfg['img_size'],
            'in_channels': 1,
            'out_ch': 1,
            'ch': 128,
            'ch_mult': [1, 2, 4, 4],
            'num_res_blocks': 2,
            'attn_resolutions': [],
            'dropout': 0.0
        },
        lossconfig={"target": "torch.nn.Identity"},
        embed_dim=cfg['z_channels']
    )
    
    if 'vae' in ckpt:
        vae.load_state_dict(ckpt['vae'], strict=False)
        print("  ✓ VAE weights loaded")
    
    # 2. UNet
    print("Building UNet...")
    prev_cfg = ckpt.get('cfg', {})
    lu = prev_cfg.get('latent_unet', {})
    
    model_state = ckpt.get('model', {})
    if 'input_blocks.0.0.weight' in model_state:
        unet_in_ch = model_state['input_blocks.0.0.weight'].shape[1]
    else:
        unet_in_ch = cfg['z_channels']
    
    if 'out.2.weight' in model_state:
        unet_out_ch = model_state['out.2.weight'].shape[0]
    else:
        unet_out_ch = cfg['z_channels']
    
    print(f"  UNet: in_channels={unet_in_ch}, out_channels={unet_out_ch}")
    
    unet = UNetModel(
        image_size=32,
        in_channels=unet_in_ch,
        out_channels=unet_out_ch,
        model_channels=lu.get('model_channels', 192),
        num_res_blocks=lu.get('num_res_blocks', 2),
        channel_mult=tuple(lu.get('channel_mult', (1, 2, 3, 4))),
        attention_resolutions=(16,),
        num_heads=lu.get('num_heads', -1),
        num_head_channels=lu.get('num_head_channels', 64),
        dropout=lu.get('dropout', 0.1),
        use_scale_shift_norm=True,
        use_checkpoint=False,
    )
    
    if 'model' in ckpt:
        unet.load_state_dict(ckpt['model'], strict=True)
        print("  ✓ UNet weights loaded")
    
    # 3. Diffusion schedule
    print("Building diffusion schedule...")
    betas = torch.tensor(
        make_beta_schedule(cfg['beta_schedule'], cfg['num_timesteps']),
        dtype=torch.float32
    )
    alphas = 1.0 - betas
    a_bar = torch.cumprod(alphas, dim=0)
    
    schedule = {
        'betas': betas.to(device),
        'alphas': alphas.to(device),
        'a_bar': a_bar.to(device),
        'a_bar_prev': torch.cat([torch.ones(1), a_bar[:-1]]).to(device)
    }
    print("  ✓ Schedule created")
    
    # 4. Build Sequential Refinement model
    model = SequentialRefinementLDM(
        vae=vae,
        unet=unet,
        schedule=schedule,
        num_selected=cfg['num_selected'],
        ddim_steps=cfg['ddim_steps'],
        ddim_eta=cfg['ddim_eta']
    ).to(device)
    
    return model


# Build model
print("="*70)
model = build_sequential_refinement_model(CFG)
print("="*70)
print("\n✓ Sequential Refinement LDM built successfully!\n")


Loading pretrained 2D LDM from: ./ckpt_ct2mri_b0_ldm_strip/last.pt
Building VAE...
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Building UNet...
  UNet: in_channels=4, out_channels=3
  ✓ UNet weights loaded
Building diffusion schedule...
  ✓ Schedule created

Building Sequential Refinement LDM...
  ✓ Stage 1: Frozen LDM (VAE + UNet)
  UNet output channels: 3
  Adding output projection: 3 → 4
  ✓ Quality Selector (Learnable)
  ✓ Context Encoder
  ✓ Attention-based Refiner

Model parameters:
  Trainable: 33,691,429 (11.0%)
  Total: 305,844,381

✓ Sequential Refinement LDM built successfully!



## 7. Training

In [12]:
# Datasets
print("Preparing datasets...")
train_ds = SequentialRefinementDataset(
    data_root=CFG["data_root"],
    context_range=CFG["context_range"],
    img_size=CFG["img_size"],
    mode='train',
    train_ratio=CFG["train_ratio"]
)

val_ds = SequentialRefinementDataset(
    data_root=CFG["data_root"],
    context_range=CFG["context_range"],
    img_size=CFG["img_size"],
    mode='val',
    train_ratio=CFG["train_ratio"]
)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"]
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Preparing datasets...

[TRAIN] Collecting patient slices from ../data/processed_with_skullstrip...
  Found 360 patients
  1973 samples from 324 patients
  Context range: ±5 slices


[VAL] Collecting patient slices from ../data/processed_with_skullstrip...
  Found 360 patients
  204 samples from 36 patients
  Context range: ±5 slices


Train batches: 123
Val batches: 13


In [13]:
# Loss function
criterion = RefinementLoss().to(CFG["device"])

# Optimizer (only trainable params)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

# Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"],
    eta_min=1e-6
)

print("✓ Training setup complete!")

✓ RefinementLoss with VGG16 perceptual loss
✓ Training setup complete!


In [ ]:
# Training loop
best_val_loss = float('inf')
history = {
    'train_loss': [], 'val_loss': [],
    'train_recon': [], 'val_recon': [],
    'train_quality': [], 'val_quality': [],
    'train_perc': [], 'val_perc': [],
}

print("\n" + "="*70)
print("Starting Training")
print("="*70)

for epoch in range(CFG["epochs"]):
    
    # Print phase info
    if epoch == 0:
        print(f"\n[Phase 1] Epochs 0-{CFG['phase1_epochs']}: Reconstruction only")
    elif epoch == CFG['phase1_epochs']:
        print(f"\n[Phase 2] Epochs {CFG['phase1_epochs']}-{CFG['phase2_epochs']}: + Perceptual")
    elif epoch == CFG['phase2_epochs']:
        print(f"\n[Phase 3] Epochs {CFG['phase2_epochs']}-{CFG['epochs']}: Full loss")
    
    # ===== Training =====
    model.train()
    # Keep Stage 1 in eval mode
    model.vae.eval()
    model.unet.eval()
    
    train_losses = []
    train_metrics = {'recon': [], 'quality': [], 'perc': []}
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']} [Train]")
    
    for batch in pbar:
        ct_seq = batch['ct_sequence'].to(CFG["device"])  # (B, 11, 1, 256, 256)
        mri_target = batch['mri_center'].to(CFG["device"])  # (B, 1, 256, 256)
        
        optimizer.zero_grad()
        
        # Forward
        refined, draft, quality_loss = model(ct_seq, mri_target)
        
        # Compute loss
        loss_dict = criterion(refined, draft, mri_target, quality_loss, epoch)
        
        loss = loss_dict['total']
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        
        optimizer.step()
        
        # Record
        train_losses.append(loss.item())
        train_metrics['recon'].append(loss_dict['recon'].item())
        train_metrics['quality'].append(loss_dict['quality'].item())
        train_metrics['perc'].append(loss_dict['perceptual'].item())
        
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'recon': f"{loss_dict['recon'].item():.4f}"
        })
    
    scheduler.step()
    
    # ===== Validation =====
    model.eval()
    val_losses = []
    val_metrics = {'recon': [], 'quality': [], 'perc': []}
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']} [Val]"):
            ct_seq = batch['ct_sequence'].to(CFG["device"])
            mri_target = batch['mri_center'].to(CFG["device"])
            
            refined, draft, quality_loss = model(ct_seq, mri_target)
            
            loss_dict = criterion(refined, draft, mri_target, quality_loss, epoch)
            
            val_losses.append(loss_dict['total'].item())
            val_metrics['recon'].append(loss_dict['recon'].item())
            val_metrics['quality'].append(loss_dict['quality'].item())
            val_metrics['perc'].append(loss_dict['perceptual'].item())
    
    # Compute averages
    avg_train_loss = np.mean(train_losses)
    avg_val_loss = np.mean(val_losses)
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_recon'].append(np.mean(train_metrics['recon']))
    history['val_recon'].append(np.mean(val_metrics['recon']))
    history['train_quality'].append(np.mean(train_metrics['quality']))
    history['val_quality'].append(np.mean(val_metrics['quality']))
    history['train_perc'].append(np.mean(train_metrics['perc']))
    history['val_perc'].append(np.mean(val_metrics['perc']))
    
    # Print summary
    print(f"\nEpoch {epoch+1}/{CFG['epochs']}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss:   {avg_val_loss:.4f}")
    print(f"    ├─ Recon:  {np.mean(val_metrics['recon']):.4f}")
    print(f"    ├─ Quality: {np.mean(val_metrics['quality']):.4f}")
    print(f"    └─ Percept: {np.mean(val_metrics['perc']):.4f}")
    print(f"  LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'config': CFG,
            'val_loss': avg_val_loss,
            'history': history
        }, Path(CFG["save_dir"]) / 'best_model.pt')
        print(f"  ✓ Saved best model (val_loss: {avg_val_loss:.4f})")
    
    # Save checkpoint
    if (epoch + 1) % CFG["save_every"] == 0:
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'config': CFG
        }, Path(CFG["save_dir"]) / f'checkpoint_epoch{epoch+1}.pt')
        print(f"  ✓ Saved checkpoint")
    
    print("="*70)

print("\n🎉 Training complete!")


Starting Training

[Phase 1] Epochs 0-20: Reconstruction only


Epoch 1/100 [Val]: 100%|██████████| 13/13 [02:21<00:00, 10.88s/it]



Epoch 1/100
  Train Loss: 0.3347
  Val Loss:   0.1080
    ├─ Recon:  0.1080
    ├─ Quality: 0.0000
    └─ Percept: 0.0000
  LR: 0.000100
  ✓ Saved best model (val_loss: 0.1080)


Epoch 2/100 [Val]: 100%|██████████| 13/13 [02:21<00:00, 10.87s/it]



Epoch 2/100
  Train Loss: 0.0848
  Val Loss:   0.0998
    ├─ Recon:  0.0998
    ├─ Quality: 0.0000
    └─ Percept: 0.0000
  LR: 0.000100
  ✓ Saved best model (val_loss: 0.0998)


Epoch 3/100 [Val]: 100%|██████████| 13/13 [02:21<00:00, 10.88s/it]



Epoch 3/100
  Train Loss: 0.0698
  Val Loss:   0.0644
    ├─ Recon:  0.0644
    ├─ Quality: 0.0000
    └─ Percept: 0.0000
  LR: 0.000100
  ✓ Saved best model (val_loss: 0.0644)


Epoch 4/100 [Val]: 100%|██████████| 13/13 [02:21<00:00, 10.90s/it]



Epoch 4/100
  Train Loss: 0.0645
  Val Loss:   0.0623
    ├─ Recon:  0.0623
    ├─ Quality: 0.0000
    └─ Percept: 0.0000
  LR: 0.000100
  ✓ Saved best model (val_loss: 0.0623)


Epoch 5/100 [Train]:  48%|████▊     | 59/123 [10:58<11:53, 11.14s/it, loss=0.0635, recon=0.0635]

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Reconstruction loss
axes[0, 1].plot(history['train_recon'], label='Train', linewidth=2)
axes[0, 1].plot(history['val_recon'], label='Val', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Reconstruction Loss')
axes[0, 1].set_title('Reconstruction Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Quality loss
axes[1, 0].plot(history['train_quality'], label='Train', linewidth=2)
axes[1, 0].plot(history['val_quality'], label='Val', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Quality Loss')
axes[1, 0].set_title('Quality Selector Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Perceptual loss
axes[1, 1].plot(history['train_perc'], label='Train', linewidth=2)
axes[1, 1].plot(history['val_perc'], label='Val', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Perceptual Loss')
axes[1, 1].set_title('Perceptual Loss (VGG)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(Path(CFG["save_dir"]) / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest validation loss: {best_val_loss:.4f}")

## 8. Inference & Visualization

In [ ]:
# Load best model
print("Loading best model...")
ckpt = torch.load(Path(CFG["save_dir"]) / 'best_model.pt', map_location=CFG["device"])
model.load_state_dict(ckpt['model'])
model.eval()
print(f"✓ Loaded model from epoch {ckpt['epoch']+1} (val_loss: {ckpt['val_loss']:.4f})")

In [ ]:
# Inference on validation samples
print("\nRunning inference...")

n_samples = 8
val_samples = [val_ds[i] for i in range(0, min(n_samples, len(val_ds)), max(1, len(val_ds)//n_samples))]

results = []

with torch.no_grad():
    for sample in tqdm(val_samples, desc="Generating predictions"):
        ct_seq = sample['ct_sequence'].unsqueeze(0).to(CFG["device"])  # (1, 11, 1, 256, 256)
        mri_target = sample['mri_center'].unsqueeze(0).to(CFG["device"])  # (1, 1, 256, 256)
        
        # Full inference with intermediates
        refined, draft, quality_loss, selected_indices = model(
            ct_seq, 
            mri_target,
            return_intermediates=True
        )
        
        results.append({
            'ct_center': ct_seq[0, 5].cpu(),  # Center CT
            'draft': draft.cpu(),
            'refined': refined.cpu(),
            'target': mri_target.cpu(),
            'selected_indices': selected_indices.cpu(),
            'patient': sample['patient'],
            'slice_idx': sample['slice_idx']
        })

print(f"✓ Generated {len(results)} predictions")

In [ ]:
# Visualize results
fig, axes = plt.subplots(len(results), 4, figsize=(16, 4*len(results)))
if len(results) == 1:
    axes = axes.reshape(1, -1)

for i, res in enumerate(results):
    # CT
    ct_img = ((res['ct_center'][0] + 1) * 0.5).numpy()
    axes[i, 0].imshow(ct_img, cmap='gray')
    axes[i, 0].set_title(f"CT Input\n{res['patient']} slice {res['slice_idx']}")
    axes[i, 0].axis('off')
    
    # Draft
    draft_img = ((res['draft'][0, 0] + 1) * 0.5).clamp(0, 1).numpy()
    axes[i, 1].imshow(draft_img, cmap='gray')
    axes[i, 1].set_title('Draft MRI (Stage 1)')
    axes[i, 1].axis('off')
    
    # Refined
    refined_img = ((res['refined'][0, 0] + 1) * 0.5).clamp(0, 1).numpy()
    axes[i, 2].imshow(refined_img, cmap='gray')
    selected_str = ', '.join(map(str, res['selected_indices'][0].tolist()))
    axes[i, 2].set_title(f'Refined MRI (Stage 2)\nSelected: [{selected_str}]')
    axes[i, 2].axis('off')
    
    # Target
    target_img = ((res['target'][0, 0] + 1) * 0.5).numpy()
    axes[i, 3].imshow(target_img, cmap='gray')
    
    # Compute metrics
    draft_mse = F.mse_loss(res['draft'], res['target']).item()
    refined_mse = F.mse_loss(res['refined'], res['target']).item()
    improvement = (draft_mse - refined_mse) / draft_mse * 100
    
    axes[i, 3].set_title(
        f'Ground Truth\n'
        f'Draft MSE: {draft_mse:.4f}\n'
        f'Refined MSE: {refined_mse:.4f}\n'
        f'Improvement: {improvement:.1f}%'
    )
    axes[i, 3].axis('off')

plt.tight_layout()
plt.savefig(Path(CFG["save_dir"]) / 'inference_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization complete!")

## 9. Summary

In [ ]:
print("="*70)
print("🎉 Sequential Quality-Aware Refinement LDM Complete!")
print("="*70)
print(f"\nCheckpoints saved in: {CFG['save_dir']}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"\n✨ Key achievements:")
print(f"  1. ✅ Stage 1 (Frozen LDM) generates drafts")
print(f"  2. ✅ Learnable quality selector picks top-{CFG['num_selected']} drafts")
print(f"  3. ✅ Attention U-Net refines with context")
print(f"  4. ✅ Progressive training (3 phases)")
print(f"  5. ✅ Sequential information fully utilized")
print("="*70)

# Compute average improvement
improvements = []
for res in results:
    draft_mse = F.mse_loss(res['draft'], res['target']).item()
    refined_mse = F.mse_loss(res['refined'], res['target']).item()
    improvement = (draft_mse - refined_mse) / draft_mse * 100
    improvements.append(improvement)

print(f"\n📊 Average improvement over draft: {np.mean(improvements):.2f}%")
print(f"   Best improvement: {np.max(improvements):.2f}%")
print(f"   Worst improvement: {np.min(improvements):.2f}%")